# 6. Classified Spend Analytics

Convenience SQL views joining `cat_predictions` to `invoices` so
downstream BI tools (Lakeview, Genie, ad-hoc SQL) have one place to
query categorized spend across any taxonomy.

Views created (one per registered schema):
- `vw_spend_by_<schema>` — invoice rows joined to their predicted code
  for that schema. When multiple sources have classified the same
  invoice (e.g. `ai_classify` + `agent_review`), the highest-confidence
  prediction wins; ties broken by `classified_at`.

Confidence is now normalized to a single 0-1 scale across all sources
(see `src/confidence.py`), so `ROW_NUMBER OVER (ORDER BY confidence)`
ranks them meaningfully.

In [ ]:
%pip install pyyaml openai openpyxl pydantic
%restart_python

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_spark
from src.config import load_config
from src.schemas import list_schemas

spark = get_spark()
config = load_config()

dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
spark.sql(f'USE {catalog}.{schema}')

In [ ]:
for s in list_schemas():
    name = s.name
    spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.{schema}.vw_spend_by_{name} AS
    WITH best AS (
      SELECT order_id, code, label, level_path, source, confidence,
             ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY COALESCE(confidence, 0) DESC, classified_at DESC
             ) AS r
      FROM {catalog}.{schema}.cat_predictions
      WHERE schema_name = '{name}'
    )
    SELECT i.order_id, i.date, i.supplier, i.region, i.cost_centre,
           i.total AS amount,
           b.code, b.label, b.level_path, b.source AS classifier,
           b.confidence
    FROM {catalog}.{schema}.invoices i
    LEFT JOIN best b ON b.order_id = i.order_id AND b.r = 1
    """)
    print(f'Created view vw_spend_by_{name}')

spark.sql(f"SHOW VIEWS IN {catalog}.{schema} LIKE 'vw_spend_by_*'").display()

In [ ]:
spark.sql(f"SELECT * FROM {catalog}.{schema}.vw_spend_by_three_level LIMIT 10").display()